# Getting Started with Agenkit

Welcome to Agenkit! This interactive tutorial will guide you through the fundamentals of building AI agents with Agenkit.

## What You'll Learn

1. **Installation** - Setting up Agenkit
2. **Your First Agent** - Creating a simple echo agent
3. **Composing Agents** - Building agent pipelines
4. **Working with LLMs** - Connecting to OpenAI/Claude
5. **Running and Testing** - Executing and validating agents

## Prerequisites

- Python 3.9+
- Basic Python knowledge
- (Optional) OpenAI or Anthropic API key

Let's get started! 🚀

> **Note**: This tutorial is also available as a [Marimo notebook](01-getting-started.py) for a reactive experience!

## 1. Installation

First, let's install Agenkit. If you're running this notebook, you likely already have it installed, but here's how you would install it:

In [ ]:
# Install Agenkit (uncomment to run)
# !pip install agenkit

# Verify installation
import agenkit
print(f"✅ Agenkit version: {agenkit.__version__}")

## 2. Your First Agent: Echo Agent

Let's create the simplest possible agent - one that echoes back whatever you say to it.

### Core Concepts

- **Agent**: Implements the `Agent` interface with `name()` and `process()` methods
- **Message**: Universal message format for agent communication
- **Async**: Agents use async/await for non-blocking operations

In [ ]:
from agenkit import Agent, Message

class EchoAgent(Agent):
    """Simple agent that echoes messages back."""
    
    def name(self) -> str:
        return "echo-agent"
    
    def capabilities(self) -> list[str]:
        return ["echo", "simple"]
    
    async def process(self, message: Message) -> Message:
        # Extract the text content from the message
        user_text = message.content
        
        # Create a response message
        response = Message(
            role="assistant",
            content=f"Echo: {user_text}"
        )
        
        return response

# Create an instance of our agent
agent = EchoAgent()
print(f"✅ Created agent: {agent.name()}")
print(f"   Capabilities: {agent.capabilities()}")

### Testing the Echo Agent

Let's send a message to our agent and see how it responds:

In [ ]:
import asyncio

# Create a message
message = Message(
    role="user",
    content="Hello, Agenkit!"
)

# Process the message
response = await agent.process(message)

print("User:", message.content)
print("Agent:", response.content)

### Interactive: Try It Yourself!

Modify the cell below to send different messages to the agent:

In [ ]:
# Try sending your own message!
my_message = Message(role="user", content="YOUR MESSAGE HERE")
my_response = await agent.process(my_message)

print(f"You: {my_message.content}")
print(f"Echo Agent: {my_response.content}")

## 3. Composing Agents: Sequential Pipeline

The real power of Agenkit comes from composing agents into pipelines. Let's create a simple pipeline with multiple agents.

### Creating a Counter Agent

First, let's create another simple agent that counts words:

In [ ]:
class WordCounterAgent(Agent):
    """Agent that counts words in a message."""
    
    def name(self) -> str:
        return "word-counter"
    
    def capabilities(self) -> list[str]:
        return ["word-count", "analysis"]
    
    async def process(self, message: Message) -> Message:
        text = message.content
        word_count = len(text.split())
        
        response = Message(
            role="assistant",
            content=f"{text} (Word count: {word_count})",
            metadata={"word_count": word_count}
        )
        
        return response

counter = WordCounterAgent()
print(f"✅ Created agent: {counter.name()}")

### Building a Sequential Pipeline

Now let's compose these agents into a pipeline using `SequentialAgent`:

In [ ]:
from agenkit.composition import SequentialAgent

# Create a pipeline: Echo first, then count words
pipeline = SequentialAgent([
    EchoAgent(),
    WordCounterAgent()
])

print(f"✅ Created pipeline: {pipeline.name()}")
print(f"   Agents: {len(pipeline.agents)}")

### Testing the Pipeline

Let's send a message through the entire pipeline:

In [ ]:
message = Message(
    role="user",
    content="Agenkit makes building agents easy"
)

result = await pipeline.process(message)

print("Input:", message.content)
print("Output:", result.content)
print("Metadata:", result.metadata)

### Visualizing the Pipeline Flow

Let's trace what happens at each step:

In [ ]:
# Process step-by-step to see the flow
message = Message(role="user", content="Hello world")

print("🔵 Step 1: Input")
print(f"   {message.content}")
print()

# First agent (Echo)
step1 = await EchoAgent().process(message)
print("🔵 Step 2: After Echo Agent")
print(f"   {step1.content}")
print()

# Second agent (Counter)
step2 = await WordCounterAgent().process(step1)
print("🔵 Step 3: After Word Counter Agent")
print(f"   {step2.content}")
print(f"   Metadata: {step2.metadata}")

## 4. Working with LLMs

Now let's connect to a real LLM! We'll create agents that use OpenAI and Anthropic.

### Setting Up API Keys

You'll need an API key from OpenAI or Anthropic. Set it as an environment variable:

In [ ]:
import os

# Check if keys are set
has_openai = bool(os.getenv("OPENAI_API_KEY"))
has_anthropic = bool(os.getenv("ANTHROPIC_API_KEY"))

print(f"OpenAI API Key: {'✅ Set' if has_openai else '❌ Not set'}")
print(f"Anthropic API Key: {'✅ Set' if has_anthropic else '❌ Not set'}")
print()
print("If you don't have API keys, you can skip this section.")

### Creating an OpenAI Agent

Let's create an agent that uses GPT-4:

In [ ]:
# Only run if you have an OpenAI API key
if has_openai:
    from agenkit.llm import OpenAIAdapter
    from agenkit.patterns import ConversationalAgent
    
    # Create LLM adapter
    llm = OpenAIAdapter(
        api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-4"
    )
    
    # Create conversational agent
    gpt4_agent = ConversationalAgent(
        llm=llm,
        system_prompt="You are a helpful AI assistant. Be concise and friendly."
    )
    
    print("✅ Created GPT-4 agent")
    
    # Test it
    response = await gpt4_agent.process(
        Message(role="user", content="What is Agenkit in one sentence?")
    )
    
    print("\nAgent response:")
    print(response.content)
else:
    print("⚠️  OpenAI API key not set - skipping this example")

### Creating an Anthropic Agent

Similarly, let's create an agent using Claude:

In [ ]:
# Only run if you have an Anthropic API key
if has_anthropic:
    from agenkit.llm import AnthropicAdapter
    from agenkit.patterns import ConversationalAgent
    
    # Create LLM adapter
    llm = AnthropicAdapter(
        api_key=os.getenv("ANTHROPIC_API_KEY"),
        model="claude-3-5-sonnet-20241022"
    )
    
    # Create conversational agent
    claude_agent = ConversationalAgent(
        llm=llm,
        system_prompt="You are a helpful AI assistant. Be concise and friendly."
    )
    
    print("✅ Created Claude agent")
    
    # Test it
    response = await claude_agent.process(
        Message(role="user", content="What is Agenkit in one sentence?")
    )
    
    print("\nAgent response:")
    print(response.content)
else:
    print("⚠️  Anthropic API key not set - skipping this example")

## 5. Running and Testing

Let's explore different ways to run and test agents.

### Multi-turn Conversations

The `ConversationalAgent` maintains context across multiple messages:

In [ ]:
if has_openai or has_anthropic:
    # Pick whichever agent you created
    conversation_agent = gpt4_agent if has_openai else claude_agent
    
    # First message
    msg1 = await conversation_agent.process(
        Message(role="user", content="My name is Alice")
    )
    print("Turn 1:")
    print(f"  User: My name is Alice")
    print(f"  Agent: {msg1.content}")
    print()
    
    # Second message - agent should remember the name
    msg2 = await conversation_agent.process(
        Message(role="user", content="What's my name?")
    )
    print("Turn 2:")
    print(f"  User: What's my name?")
    print(f"  Agent: {msg2.content}")
else:
    print("⚠️  No LLM API keys set - skipping conversation example")

### Error Handling

Let's see how to handle errors gracefully:

In [ ]:
class SafeAgent(Agent):
    """Agent with error handling."""
    
    def name(self) -> str:
        return "safe-agent"
    
    async def process(self, message: Message) -> Message:
        try:
            # Simulate processing
            if "error" in message.content.lower():
                raise ValueError("Simulated error for demonstration")
            
            return Message(
                role="assistant",
                content=f"Processed: {message.content}"
            )
        except Exception as e:
            # Handle error gracefully
            return Message(
                role="assistant",
                content=f"Error occurred: {str(e)}",
                metadata={"error": True, "error_type": type(e).__name__}
            )

# Test error handling
safe_agent = SafeAgent()

# Success case
response1 = await safe_agent.process(Message(role="user", content="Hello"))
print("✅ Success case:")
print(f"   {response1.content}")
print()

# Error case
response2 = await safe_agent.process(Message(role="user", content="Trigger error"))
print("❌ Error case:")
print(f"   {response2.content}")
print(f"   Metadata: {response2.metadata}")

### Testing with Assertions

Here's how you might write tests for your agents:

In [ ]:
async def test_echo_agent():
    """Test that echo agent works correctly."""
    agent = EchoAgent()
    
    # Test 1: Basic echoing
    msg = Message(role="user", content="test")
    response = await agent.process(msg)
    assert "Echo: test" in response.content
    print("✅ Test 1 passed: Basic echoing")
    
    # Test 2: Role is correct
    assert response.role == "assistant"
    print("✅ Test 2 passed: Correct role")
    
    # Test 3: Agent name
    assert agent.name() == "echo-agent"
    print("✅ Test 3 passed: Correct name")
    
    print("\n🎉 All tests passed!")

# Run tests
await test_echo_agent()

## Summary

Congratulations! 🎉 You've learned the fundamentals of Agenkit:

✅ **Installation** - Set up Agenkit in your environment  
✅ **First Agent** - Created a simple echo agent  
✅ **Composition** - Built pipelines with multiple agents  
✅ **LLM Integration** - Connected to OpenAI and Anthropic  
✅ **Testing** - Wrote tests and handled errors  

## Next Steps

Ready to dive deeper? Check out:

- **[Tutorial 02: Production Patterns](02-production-patterns.ipynb)** - Learn middleware, observability, and error handling
- **[Tutorial 03: Advanced Reasoning](03-advanced-reasoning.ipynb)** - Explore Chain-of-Thought, Tree-of-Thought, and more
- **[Marimo Version](01-getting-started.py)** - Try the reactive notebook version
- **[Examples Directory](https://github.com/scttfrdmn/agenkit/tree/main/examples)** - 150+ examples across all patterns
- **[API Documentation](https://agenkit.dev/api/)** - Complete API reference

## Resources

- 📚 [Documentation](https://agenkit.dev)
- 💬 [GitHub Discussions](https://github.com/scttfrdmn/agenkit/discussions)
- 🐛 [Report Issues](https://github.com/scttfrdmn/agenkit/issues)
- ⭐ [Star on GitHub](https://github.com/scttfrdmn/agenkit)

Happy building! 🚀